# Planowanie linii metra we Wrocławiu

Notebook jest szkicem projektu badawczo-inżynierskiego: chcemy znaleźć wariant przebiegu metra, który przy długości i liczbie stacji jak warszawska M1 przechodzi przez centrum Wrocławia, unika obszarów zalewowych i obsługuje jak największy popyt.

Założenie bazowe: jedna linia ma 23,1 km i 21 stacji. Scenariusze `1`, `2`, `3` oznaczają odpowiednio sieć z jedną, dwiema i trzema liniami. Koszt kilometra, promień dojścia do stacji, kara za obszary zalewowe i kara za nadwyżkę geologiczną są parametrami, które można zmieniać.

## Logika projektu

1. Zbieramy popyt: najlepiej z demografii SIP Wrocławia, a alternatywnie z lokali wyborczych i liczby oddanych głosów.
2. Zamieniamy popyt na ważone punkty: centroid/reprezentatywny punkt regionu, adresu albo obwodu.
3. Wczytujemy obszary zakazane: przede wszystkim realne obszary zagrożenia powodziowego MZP/ISOK, jeśli są dostępne lokalnie. Rzeki i wody powierzchniowe są osobną warstwą bariery komunikacyjnej, nie zakazem dla metra.
4. Wczytujemy geologię/litologię jako `cost_factor` i zamieniamy przebieg przez trudniejszy teren na `geology_excess_km`.
5. Popyt leżący w obszarze zakazanym przesuwamy do najbliższego wolnego miejsca, bez zmiany wagi.
6. Tworzymy centra regionalne i kandydatów na stacje.
7. Formułujemy problem jako orienteering / prize-collecting TSP: wybierz i uporządkuj kandydatów tak, żeby zmieścić się w 23,1 km i zebrać jak najwięcej popytu.
8. Rozwiązujemy problem heurystyką greedy insertion + 2-opt, a warianty oceniamy po obsłużonym popycie, przejściu przez strefy zakazane, geologii i koszcie.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd


import wro_metro as wm

pd.set_option("display.max_columns", 100)
plt.rcParams["figure.dpi"] = 120

PROJECT_ROOT

## Źródła danych

Najbardziej wiarygodny wariant popytu to demografia SIP Wrocławia, bo jest przestrzenna i aktualna. Lokale wyborcze są dobrym proxy aktywności, ale nie powinny być jedyną miarą populacji: głosy zależą od frekwencji, wieku, granic obwodów i typu wyborów.


In [ ]:
sources = pd.DataFrame([
    {
        "warstwa": "Benchmark metra",
        "zastosowanie": "23,1 km i 21 stacji jako ograniczenia bazowe",
        "url": "https://metro.waw.pl/metro-warszawskie/linia-m1/przebieg/",
    },
    {
        "warstwa": "Demografia SIP Wrocławia 1998-2025",
        "zastosowanie": "najlepszy bazowy popyt: ludność według rejonów statystycznych i urbanistycznych",
        "url": "https://geoportal.wroclaw.pl/zasoby/",
    },
    {
        "warstwa": "Granice osiedli i adresy SIP",
        "zastosowanie": "centra osiedli, agregacje, walidacja adresów",
        "url": "https://geoportal.wroclaw.pl/zasoby/",
    },
    {
        "warstwa": "Wody powierzchniowe SIP",
        "zastosowanie": "tymczasowe proxy przeszkód wodnych, nie zamiennik map zalewowych",
        "url": "https://geoportal.wroclaw.pl/zasoby/",
    },
    {
        "warstwa": "Mapy zagrożenia powodziowego ISOK/Wody Polskie",
        "zastosowanie": "obszary zakazane albo silnie karane w przebiegu trasy",
        "url": "https://wody.isok.gov.pl/imap_kzgw/?gpmap=gpMZP",
    },
    {
        "warstwa": "Mapa Litogenetyczna Polski 1:50 000 PIG-PIB/CBDG",
        "zastosowanie": "warstwa litologii zamieniana na heurystyczny cost_factor dla trudnosci budowy",
        "url": "https://cbdgmapa.pgi.gov.pl/arcgis/rest/services/kartografia/mlp50k/MapServer/6",
    },
    {
        "warstwa": "GUS BDL API",
        "zastosowanie": "dane kontrolne i agregaty demograficzne, mniej szczegółowe przestrzennie",
        "url": "https://api.stat.gov.pl/Home/BdlApi",
    },
])
sources

## Wczytanie danych

Notebook najpierw próbuje użyć oficjalnych paczek SIP Wrocławia z `data/raw`. Jeżeli ich nie ma, przełącza się na mały zestaw demonstracyjny. Popyt rysujemy jako obszary demograficzne, a algorytm używa ich punktów reprezentatywnych tylko jako technicznej reprezentacji do liczenia odległości. Dane zalewowe z ISOK najlepiej dodać jako `data/raw/flood_zones.geojson`. Wody powierzchniowe nie są traktowane jako zakaz dla metra: są osobną warstwą bariery komunikacyjnej, której przecięcie może poprawić połączenia między brzegami.


In [ ]:
config = wm.MetroConfig(
    cost_per_km_mln=650.0,
    flood_cost_multiplier=2.0,
    walk_radius_m=1_100.0,
    angle_step_deg=2.0,
    forbidden_penalty_per_km=60_000.0,
    outside_city_penalty_per_km=120_000.0,
    geology_penalty_per_km=40_000.0,
    high_geology_factor_threshold=1.50,
    high_geology_penalty_per_km=120_000.0,
    anchor_geology_penalty_per_excess=35_000.0,
    anchor_high_geology_penalty=25_000.0,
    station_geology_penalty_per_excess=12_000.0,
    station_high_geology_penalty=8_000.0,
    station_geology_score_factor=1.40,
    river_crossing_bonus_per_km=600.0,
    transfer_bonus_per_interchange=35_000.0,
    interchange_radius_m=450.0,
    line_overlap_penalty_per_km=220_000.0,
    parallel_line_buffer_m=900.0,
    residual_coverage_multiplier=1.35,
    station_risk_buffer_m=260.0,
    candidate_catchment_radius_m=1_600.0,
    central_anchor_radius_m=3_000.0,
    min_central_anchor_candidates=8,
    max_central_anchor_share=0.32,
    min_regional_anchor_candidates=8,
    min_directional_anchor_candidates_per_sector=5,
    route_anchor_count=8,
    max_turn_angle_deg=45.0,
    hard_max_turn_angle_deg=85.0,
    turn_penalty_per_degree=1_500.0,
    minimum_curve_radius_m=450.0,
    curve_radius_penalty_per_m=90.0,
    corridor_detour_ratio_limit=1.35,
    corridor_detour_penalty_per_ratio=90_000.0,
    corridor_backtrack_penalty_per_km=45_000.0,
    adaptive_station_placement=True,
    station_min_spacing_m=800.0,
    station_max_spacing_m=1_700.0,
    station_candidate_step_m=125.0,
    station_flood_score_factor=0.03,
    station_water_buffer_m=90.0,
    station_water_score_factor=0.02,
    station_terminal_flex_m=1_500.0,
    station_anchor_bonus_radius_m=260.0,
    station_anchor_bonus=7_500.0,
)

raw_dir = PROJECT_ROOT / "data" / "raw"
demography_zip = raw_dir / "dem-rejurb-rejstat-shp.zip"

if demography_zip.exists():
    population_layer = wm.load_wroclaw_demography(raw_dir, year=2025, unit="REJSTAT")
    demand_areas = wm.demand_areas_from_polygons(population_layer, weight_col="SUMA")
    demand_raw = wm.to_demand_points(population_layer, weight_col="SUMA")
    demand_raw["name"] = "rejon_" + demand_raw["REJON"].astype(str)
    osiedla = wm.load_wroclaw_osiedla(raw_dir)
    centre_area = osiedla[osiedla["NAZWAOSIED"].eq("Stare Miasto")].copy()
    data_mode = "oficjalna demografia SIP Wrocławia 2025"
else:
    demand_raw = wm.demo_demand()
    demand_areas = gpd.GeoDataFrame(geometry=[], crs=demand_raw.crs)
    centre_area = gpd.GeoDataFrame(geometry=[], crs=demand_raw.crs)
    osiedla = gpd.GeoDataFrame(geometry=[], crs=demand_raw.crs)
    data_mode = "dane demonstracyjne"

centres = wm.demo_centres()
if not centre_area.empty:
    stare_miasto_point = centre_area.geometry.iloc[0].representative_point()
    mask = centres["role"].eq("required_city_centre")
    centres.loc[mask, "name"] = "Stare Miasto"
    centres.loc[mask, "geometry"] = [stare_miasto_point]

if not osiedla.empty:
    city_boundary = wm.city_boundary_from_layer(osiedla, name="Wroclaw boundary from SIP districts")
    city_boundary_mode = "granice osiedli SIP"
elif not demand_areas.empty:
    city_boundary = wm.city_boundary_from_layer(demand_areas, name="Wroclaw proxy from demand areas")
    city_boundary_mode = "proxy z rejonow demograficznych"
else:
    city_boundary = gpd.GeoDataFrame(
        {"name": ["demo city boundary"]},
        geometry=[demand_raw.geometry.unary_union.convex_hull.buffer(3_500)],
        crs=demand_raw.crs,
    )
    city_boundary_mode = "demo convex hull popytu + bufor"

forbidden_from_raw = wm.load_forbidden_zones_from_raw(raw_dir)
if forbidden_from_raw is not None:
    flood_zones = forbidden_from_raw
    forbidden_mode = sorted(flood_zones["risk"].dropna().unique().tolist())
else:
    flood_zones = gpd.GeoDataFrame({"risk": []}, geometry=[], crs=demand_raw.crs)
    forbidden_mode = ["brak mapy zalewowej ISOK w data/raw"]

water_crossings = wm.load_water_crossing_layer(raw_dir)
if water_crossings is None:
    water_crossings = wm.demo_flood_zones()
    water_mode = "demo river proxy"
else:
    water_mode = "wody powierzchniowe SIP jako bariera/crossing layer"

geology_from_raw = wm.load_geology_cost_layer_from_raw(raw_dir)
if geology_from_raw is not None:
    geology = geology_from_raw
    geology_mode = "realna warstwa kosztu/geologii z data/raw"
else:
    geology = gpd.GeoDataFrame({"cost_factor": []}, geometry=[], crs=demand_raw.crs)
    geology_mode = "brak realnej warstwy geologicznej; mnożnik kosztu = 1.0"

if not demand_areas.empty:
    regional_clusters = wm.regional_clusters_from_demand_areas(demand_areas, k=8)
    regional_centres = wm.regional_centres_from_clusters(regional_clusters)
else:
    regional_clusters = gpd.GeoDataFrame(geometry=[], crs=demand_raw.crs)
    regional_centres = wm.regional_centres_from_demand(demand_raw, k=8)

print(f"Tryb danych: {data_mode}")
print(f"Warstwy zakazane/proxy: {forbidden_mode}")
print(f"Warstwa rzek/wody: {water_mode}")
print(f"Granica miasta: {city_boundary_mode}")
print(f"Geologia/koszt: {geology_mode}")
print(f"Centrum wymuszone: {centres.loc[centres['role'].eq('required_city_centre'), 'name'].iloc[0]}")
print(f"Długość linii: {config.length_m/1000:.1f} km")
print(f"Liczba stacji: {config.station_count}")
print(f"Liczba kotwic korytarza: {config.route_anchor_count}")
print(f"Średni rozstaw stacji: {config.station_spacing_m:.0f} m")
print(f"Miekki limit skretu: {config.max_turn_angle_deg:.0f} deg; twardy limit: {config.hard_max_turn_angle_deg:.0f} deg")
print(f"Kara za km poza miastem: {config.outside_city_penalty_per_km:,.0f}")
print(f"Kara geologiczna za km nadwyzki kosztu: {config.geology_penalty_per_km:,.0f}")
print(f"Prog trudnej geologii: cost_factor >= {config.high_geology_factor_threshold:.2f}; kara/km: {config.high_geology_penalty_per_km:,.0f}")
print(f"Limit objazdowosci korytarza: {config.corridor_detour_ratio_limit:.2f}")
print(f"Zasieg dojscia do stacji: {config.walk_radius_m:.0f} m")
print(f"Mnoznik redukcji popytu po obsludze: {config.residual_coverage_multiplier:.2f}")

display_cols = [col for col in ["name", "REJON", "population"] if col in demand_raw.columns]
demand_raw[display_cols].sort_values("population", ascending=False).head(10)

In [ ]:
# Przykład podmiany na dane realne po pobraniu plików do data/raw:
# py -3.11 scripts\download_wroclaw_data.py
# population_layer = wm.load_wroclaw_demography(PROJECT_ROOT / "data/raw", year=2025, unit="REJSTAT")
# demand_areas = wm.demand_areas_from_polygons(population_layer, weight_col="SUMA")
# demand_raw = wm.to_demand_points(population_layer, weight_col="SUMA")
# flood_zones = wm.load_forbidden_zones_from_raw(PROJECT_ROOT / "data/raw")
# water_crossings = wm.load_water_crossing_layer(PROJECT_ROOT / "data/raw")
#
# polling = wm.load_polling_places_csv(PROJECT_ROOT / "data/raw/polling_places.csv", votes_col="votes")
# demand_raw = polling
#
# Jeżeli CSV z lokalami wyborczymi nie ma współrzędnych:
# api_key = os.environ["GOOGLE_MAPS_API_KEY"]
# polling_table = pd.read_csv(PROJECT_ROOT / "data/raw/polling_places.csv")
# polling_geocoded = wm.geocode_google_addresses(polling_table, address_col="address", api_key=api_key)
# polling_geocoded.to_csv(PROJECT_ROOT / "data/processed/polling_places_geocoded.csv", index=False)

None

### Wazne rozroznienia danych

- Obszary zalewowe: uzywamy realnej warstwy MZP/ISOK, jezeli w `data/raw` jest `flood_zones.geojson`. Mozna ja pobrac bez QGIS skryptem `py -3.11 scripts\download_flood_zones_isok.py`.
- Granica miasta: uzywamy sumy poligonow z `granice-osiedli.zip` jako `city_boundary`. Kazdy kilometr linii poza ta granica dostaje kare `outside_city_penalty_per_km`.
- Rzeka i wody powierzchniowe: pochodza z SIP Wroclawia i sa uzywane jako warstwa bariery komunikacyjnej oraz potencjalnej premii za przeprawe metrem, nie jako teren zakazany.
- Woda przy stacjach: metro moze przejsc pod rzeka albo przeciac bariere wodna, ale stacje i kotwice nie powinny wypadac na zbiornikach, rzekach ani bezposrednio w buforze ryzyka. Dlatego `water_crossings` jest warstwa unikania dla punktow stacyjnych, a nie zakazem dla tunelu.
- Warstwa geologiczna/kosztowa: notebook automatycznie wczytuje `data/raw/geology.geojson` albo inna warstwe `geology.*` / `cost_zones.*` z kolumna `cost_factor`. `cost_factor = 1.35` oznacza odcinek o 35% trudniejszy/kosztowniejszy niz neutralny grunt.
- Kara geologiczna: zamiast karac sam sredni mnoznik, liczymy `geology_excess_km`, czyli dlugosc wazona nadwyzka kosztu. 1 km przez obszar `cost_factor = 1.35` dodaje 0.35 km nadwyzki geologicznej. Dla `cost_factor >= 1.5` dochodzi osobna mocna kara `high_geology_penalty_per_km`.
- Kotwice i stacje: trudna geologia zmniejsza atrakcyjnosc kandydatow na kotwice oraz punktow, w ktorych dynamic programming moze postawic stacje. Wysoki `cost_factor` przy samej stacji lub kotwicy jest karany dodatkowo, nawet jezeli odcinek jest krotki.
- Centra regionalne: nie sa recznie wpisanymi punktami. Powstaja z klastrow wazonych populacja na obszarach demograficznych SIP, a potem kandydaci na kotwice sa korygowani o MZP, wody powierzchniowe i bufor mozliwych podtopien.
- Reprezentacja kierunkow: kandydaci na kotwice maja minimalna pule dla sektorow `south`, `north`, `east`, `west`, zeby gesto zaludnione poludnie nie znikalo tylko dlatego, ze centralne obszary maja bardzo wysoki wynik lokalny.
- Kolejne linie: po wyznaczeniu linii popyt w jej zasiegu jest mocniej redukowany (`residual_coverage_multiplier`), a wiekszy bufor/kara nakladania zniecheca druga i trzecia linie do obslugi tego samego ramienia miasta.


## Obszary popytu, zalewowość i rzeka

Założenie: popyt mieszkaniowy analizujemy jako obszary demograficzne, a do optymalizacji przekazujemy ich punkty reprezentatywne. Jeżeli punkt popytu trafi w obszar realnego ryzyka zalewowego, nie usuwamy go z modelu, tylko przesuwamy jego wagę do najbliższego wolnego miejsca. Rzeka nie jest obszarem zakazanym dla metra: potraktowana jest jako bariera komunikacyjna, której przecięcie może dać premię, bo poprawia relacje między brzegami.


In [ ]:
demand = wm.relocate_demand_from_forbidden(demand_raw, flood_zones, config=config)
if regional_clusters.empty:
    regional_centres = wm.regional_centres_from_demand(demand, k=8)

demand[["name", "population", "was_relocated", "relocated_m"]]\
    .sort_values(["was_relocated", "relocated_m"], ascending=False)\
    .head(12)

In [ ]:
if not regional_clusters.empty:
    regional_clusters[["cluster_id", "population", "area_km2", "population_density"]]\
        .sort_values("population", ascending=False)
else:
    regional_centres[["centre_id", "demand_weight"]].sort_values("demand_weight", ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
if not demand_areas.empty:
    demand_areas.plot(ax=ax, column="population_density", cmap="YlGn", alpha=0.56, edgecolor="#ffffff", linewidth=0.12, legend=True)
if not geology.empty:
    geology.plot(ax=ax, column="cost_factor", cmap="YlOrBr", alpha=0.14, edgecolor="none")
if water_crossings is not None and not water_crossings.empty:
    water_crossings.plot(ax=ax, color="#74b6d7", edgecolor="#2f6f95", alpha=0.42, linewidth=0.4)
if not flood_zones.empty:
    flood_zones.plot(ax=ax, color="#ff3b30", edgecolor="#7a1111", alpha=0.30, linewidth=0.6)
if not centre_area.empty:
    centre_area.plot(ax=ax, facecolor="none", edgecolor="#111111", linewidth=4.0, linestyle="-", label="Stare Miasto")
    centre_area.plot(ax=ax, facecolor="none", edgecolor="#ffd60a", linewidth=2.2, linestyle="-")
if not regional_clusters.empty:
    regional_clusters.plot(ax=ax, facecolor="#f72585", alpha=0.10, edgecolor="#7a1f5c", linewidth=1.2, label="klastry regionalne")
demand_raw.plot(ax=ax, color="#555555", markersize=8, alpha=0.25, label="punkty reprezentatywne obszar?w")
demand.plot(ax=ax, color="#d62828", markersize=10, alpha=0.45, label="popyt po relokacji")
regional_centres.plot(ax=ax, marker="x", color="#7a1f5c", markersize=70, linewidth=1.7, label="?rodki klastr?w")
centres.plot(ax=ax, marker="*", color="#111111", markersize=170, label="centra wymuszone")
centres.plot(ax=ax, marker="*", color="#ffd60a", edgecolor="#111111", markersize=110)

for _, row in demand[demand["was_relocated"]].iterrows():
    ax.plot(
        [row["original_geometry"].x, row.geometry.x],
        [row["original_geometry"].y, row.geometry.y],
        color="#d62828",
        linewidth=0.8,
        alpha=0.7,
    )

ax.set_title("Obszary popytu, Stare Miasto, rzeka i relokacja wag")
ax.set_axis_off()
ax.set_aspect("equal")
ax.legend(loc="lower left")
plt.show()

## Warstwy danych osobno

Poniższe mapy pokazują składniki danych osobno. Każda warstwa jest rysowana na podkładzie satelitarnym, żeby było widać realny kontekst miasta. Dla gęstości zaludnienia ciemniejszy obszar oznacza większe zagęszczenie, ale warstwa jest półprzezroczysta, więc zdjęcie satelitarne nadal zostaje czytelne.


In [ ]:
extent_for_maps = demand_areas if not demand_areas.empty else demand

layer_specs = [
    {
        "title": "Warstwa 1: gęstość zaludnienia",
        "layer": demand_areas,
        "column": "population_density",
        "cmap": "YlGn",
        "alpha": 0.54,
        "edgecolor": "none",
    },
    {
        "title": "Warstwa 2: rzeki i wody powierzchniowe jako bariera komunikacyjna",
        "layer": water_crossings,
        "column": None,
        "color": "#48bfe3",
        "alpha": 0.50,
        "linewidth": 0.45,
    },
    {
        "title": "Warstwa 3: obszary zalewowe / zakazane, jeśli dostępne",
        "layer": flood_zones,
        "column": None,
        "color": "#ff3b30",
        "alpha": 0.35,
        "linewidth": 0.75,
    },
    {
        "title": "Warstwa 4: geologia jako mnożnik kosztu",
        "layer": geology,
        "column": "cost_factor",
        "cmap": "YlOrBr",
        "alpha": 0.42,
        "edgecolor": "#f5c542",
        "linewidth": 0.8,
    },
    {
        "title": "Warstwa 5: Stare Miasto jako wymuszone centrum",
        "layer": centre_area,
        "column": None,
        "color": "#ffd60a",
        "alpha": 0.34,
        "edgecolor": "#111111",
        "linewidth": 2.4,
    },
    {
        "title": "Warstwa 6: regionalne klastry popytu",
        "layer": regional_clusters,
        "column": "population_density",
        "cmap": "magma_r",
        "alpha": 0.35,
        "edgecolor": "#ffffff",
        "linewidth": 1.2,
    },
]

for spec in layer_specs:
    fig, ax = wm.plot_data_layer(
        layer=spec["layer"],
        title=spec["title"],
        column=spec.get("column"),
        extent_layer=extent_for_maps,
        satellite=True,
        cmap=spec.get("cmap", "Greys"),
        alpha=spec.get("alpha", 0.55),
        color=spec.get("color", "#d62828"),
        edgecolor=spec.get("edgecolor", "#ffffff"),
        linewidth=spec.get("linewidth", 0.5),
        markersize=spec.get("markersize", 26),
    )
    plt.show()


## Optymalizacja wariantow jako problem NP-trudny

Wlasciwy model w projekcie to **orienteering problem**, czyli komiwojazer z nagrodami i budzetem. Kazdy kandydat na kotwice korytarza ma nagrode wynikajaca z obsluzonego popytu, kolejnosc punktow dziala jak w TSP, a limit dlugosci/kosztu dziala jak ograniczenie plecakowe. Problem jest NP-trudny, wiec w notebooku uzywamy heurystyki: greedy insertion pod limitem dlugosci, a potem lokalna poprawa 2-opt.

Funkcja celu: obsluzony popyt w promieniu dojscia do stacji, minus kara za przebieg przez obszary zalewowe, minus kara za wychodzenie poza granice Wroclawia, minus kara za trudniejsza geologie, minus kara za nakladanie sie z istniejaca linia, minus kara za ostre skrety i nienaturalny ksztalt korytarza, plus premia za sensowne przeciecie bariery rzecznej i za przesiadki z juz zaplanowanymi liniami. Obszary MZP sa dodatkowo liczone kosztowo jako drozszy odcinek budowy: `flood_cost_multiplier = 2.0`, czyli kilometr w strefie zalewowej kosztuje jak dwa zwykle kilometry. Geologia jest liczona przez `geology_excess_km`: odcinek 1 km przez `cost_factor = 1.35` dodaje 0.35 km nadwyzki kosztu i kare `0.35 * geology_penalty_per_km`. Dla `cost_factor >= 1.5` dochodzi mocna kara wysokiego ryzyka geologicznego, a trudna geologia w samym punkcie stacji/kotwicy obniza atrakcyjnosc tego punktu.

Po wybraniu korytarza 21 stacji nie jest juz rozstawiane slepo co rowny dystans. Funkcja `stations_for_corridor` wybiera punkty po korytarzu metoda dynamic programming: maksymalizuje lokalny popyt w zasiegu dojscia, pilnuje minimalnego i maksymalnego odstepu, obniza atrakcyjnosc punktow lezacych w MZP, buforze podtopien, wodach powierzchniowych albo na trudnej geologii i premiuje punkty przy kotwicach korytarza. Stacje koncowe moga przesunac sie w oknie `station_terminal_flex_m`, zeby nie zostac przypiete do ryzykownego konca korytarza.


In [ ]:
centre_point_raw = centres.loc[centres["role"].eq("required_city_centre"), "geometry"].iloc[0]

candidate_sites = wm.candidate_station_sites(
    demand=demand,
    regional_centres=regional_centres,
    centres=centres,
    forbidden=flood_zones,
    geology=geology,
    water_crossings=water_crossings,
    config=config,
    max_candidates=45,
)

centre_point = candidate_sites.loc[candidate_sites["required"], "geometry"].iloc[0]

candidate_sites[[
    "candidate_id",
    "source",
    "name",
    "base_candidate_weight",
    "catchment_weight",
    "candidate_weight",
    "required",
    "direction_sector",
    "near_required_centre",
    "anchor_relocated_m",
    "in_flood_zone",
    "in_flood_buffer",
    "distance_to_flood_m",
    "in_water_zone",
    "in_water_buffer",
    "distance_to_water_m",
    "geology_factor",
    "high_geology",
    "geology_weight_factor",
    "anchor_geology_penalty",
]].head(16)

Kandydaci na kotwice stacji s? osobn? warstw? wej?ciow? algorytmu. Teraz ich waga nie pochodzi tylko z jednego rejonu demograficznego, ale z lokalnego zasi?gu doj?cia (`candidate_catchment_radius_m`). Dzi?ki temu du?e zaludnienie wok?? centrum nie znika tylko dlatego, ?e Stare Miasto jest jedn? wymuszon? kotwic?.

Kandydaci le??cy w MZP albo w buforze ryzyka (`station_risk_buffer_m`) s? przesuwani do najbli?szego miejsca poza stref? ryzyka. Zachowujemy kolumny diagnostyczne `anchor_relocated_m`, `in_flood_zone`, `in_flood_buffer` i `distance_to_flood_m`, ?eby da?o si? opisa?, kt?re kotwice zosta?y skorygowane.

In [ ]:
fig, ax = wm.plot_data_layer(
    layer=candidate_sites,
    title="Warstwa 7: kandydaci na kotwice stacji po korekcie ryzyka",
    column="candidate_weight",
    extent_layer=extent_for_maps,
    satellite=True,
    cmap="viridis",
    alpha=0.92,
    edgecolor="#ffffff",
    markersize=54,
)

candidate_map = candidate_sites.to_crs(wm.WEB_MERCATOR)
relocated = candidate_map[candidate_map["anchor_relocated_m"].fillna(0) > 0]
required = candidate_map[candidate_map["required"]]
near_centre = candidate_map[candidate_map["near_required_centre"].fillna(False) & ~candidate_map["required"]]

if not near_centre.empty:
    ax.scatter(near_centre.geometry.x, near_centre.geometry.y, s=38, facecolors="none", edgecolors="#ffd60a", linewidths=1.2, zorder=8)
if not relocated.empty:
    ax.scatter(relocated.geometry.x, relocated.geometry.y, s=120, facecolors="none", edgecolors="#ff3b30", linewidths=2.0, zorder=9)
if not required.empty:
    ax.scatter(required.geometry.x, required.geometry.y, s=260, marker="*", c="#111111", edgecolors="#111111", linewidths=1.0, zorder=10)
    ax.scatter(required.geometry.x, required.geometry.y, s=170, marker="*", c="#ffd60a", edgecolors="#111111", linewidths=0.8, zorder=11)

plt.show()

### Formalizacja algorytmiczna

Niech `V` bedzie zbiorem kandydatow na kotwice korytarza, `c` wymaganym centrum, `I` punktami popytu, `w_i` waga popytu, a `L = 23,1 km` limitem dlugosci jednej linii. Wazna zmiana: kotwice nie sa juz pelna lista 21 stacji. To punkty kontrolne przebiegu tunelu, ktorych jest mniej (`route_anchor_count`), a stacje sa pozniej rozmieszczane adaptacyjnie wzdluz wybranego korytarza.

Decyzje modelu:

- `z_v = 1`, jezeli kandydat `v` zostal wybrany jako kotwica korytarza,
- `x_uv = 1`, jezeli trasa przechodzi bezposrednio z `u` do `v`,
- `y_i = 1` albo wartosc ciagla w `[0, 1]`, jezeli popyt `i` jest obsluzony przez finalne stacje.

Cel: maksymalizowac obsluzony popyt z karami i premiami za geometrie oraz warunki terenowe:

`max sum(w_i * y_i) - kara_zalewowa - kara_poza_miastem - kara_geologiczna - kara_nakladania - kara_skrecania + premia_rzeka + premia_przesiadki`

`kara_geologiczna = geology_excess_km * geology_penalty_per_km + high_geology_km * high_geology_penalty_per_km + kara_punktow_geologicznych`, gdzie `geology_excess_km` jest suma nadwyzki `cost_factor - 1.0` po dlugosci odcinkow tunelu. Dzieki temu 5 km przez lekko trudny grunt i 1 km przez bardzo trudny grunt nie sa traktowane tak samo, a stacje/kotwice na slabym gruncie sa karane nawet wtedy, gdy sam odcinek jest krotki.

`kara_ksztaltu = corridor_shape_penalty`, czyli kara za zbyt duzy stosunek dlugosci trasy do odleglosci miedzy koncami oraz za cofanie sie wzdluz glownej osi linii. To ma ograniczac efekt TSP, w ktorym algorytm zbiera atrakcyjne punkty zygzakiem zamiast budowac naturalne ramiona metra.

Ograniczenia: trasa musi przejsc przez Stare Miasto, musi miescic sie w budzecie dlugosci, powinna pozostawac w granicy Wroclawia, nie moze tworzyc podtras oderwanych od glownej linii, musi zachowac minimalne odstepy miedzy kotwicami i nie powinna przekraczac twardego limitu ostrego skretu. Kara skretu jest praktycznym przyblizeniem promienia skretu: im wieksze zalamanie i im krotsze sa sasiednie odcinki, tym bardziej wynik jest karany. To nadal wariant orienteering / prize-collecting TSP, wiec problem jest NP-trudny. W malej skali da sie go policzyc dokladnie, ale dla realnej liczby kandydatow potrzebujemy heurystyk.


In [ ]:
wm.complexity_growth_table([5, 8, 10, 12, 15, 20, 30], max_selected=8)

### Exact vs heurystyka na małym podzbiorze

Dla kilku kandydatów możemy zrobić pełne przeszukanie: sprawdzamy możliwe podzbiory i kolejności przejścia przez centrum. To daje punkt odniesienia. Potem na tym samym podzbiorze uruchamiamy heurystykę greedy insertion + 2-opt i porównujemy wynik.


In [ ]:
exact_vs_heuristic, small_solutions = wm.compare_exact_and_heuristic(
    demand=demand,
    candidates=candidate_sites,
    centre=centre_point,
    forbidden=flood_zones,
    geology=geology,
    config=config,
    city_boundary=city_boundary,
    max_optional_candidates=8,
    max_selected_candidates=6,
)

exact_vs_heuristic.assign(
    served_share=lambda df: (100 * df["served_share"]).round(1),
    anchor_objective_score=lambda df: df["anchor_objective_score"].round(0),
    final_score=lambda df: df["final_score"].round(0),
    anchor_score_vs_exact=lambda df: (100 * df["anchor_score_vs_exact"]).round(1),
    forbidden_km=lambda df: df["forbidden_km"].round(2),
    geology_excess_km=lambda df: df["geology_excess_km"].round(2),
    high_geology_km=lambda df: df["high_geology_km"].round(2),
    anchor_geology_penalty=lambda df: df["anchor_geology_penalty"].round(0),
    station_geology_penalty=lambda df: df["station_geology_penalty"].round(0),
    geology_factor=lambda df: df["geology_factor"].round(2),
    station_flood_zone_count=lambda df: df["station_flood_zone_count"].astype(int),
    station_water_zone_count=lambda df: df["station_water_zone_count"].astype(int),
    station_flood_buffer_count=lambda df: df["station_flood_buffer_count"].astype(int),
    station_water_buffer_count=lambda df: df["station_water_buffer_count"].astype(int),
    outside_city_km=lambda df: df["outside_city_km"].round(2),
    max_turn_angle_deg=lambda df: df["max_turn_angle_deg"].round(0),
    corridor_detour_ratio=lambda df: df["corridor_detour_ratio"].round(2),
    corridor_backtrack_km=lambda df: df["corridor_backtrack_km"].round(2),
)

`anchor_objective_score` to wynik problemu NP-trudnego liczony na wybranych kotwicach korytarza. `final_score` to wynik po rozlozeniu pelnych 21 stacji na korytarzu. Te wartosci moga sie roznic, bo kotwice steruja przebiegiem linii, a nie sa jeszcze kompletna lista stacji. Dzieki temu jedna linia nie probuje odwiedzic 21 punktow jak komiwojazer, co wczesniej dawalo nienaturalne zygzaki.


In [ ]:
scenarios = wm.build_orienteering_scenarios(
    demand=demand,
    candidates=candidate_sites,
    centre=centre_point,
    forbidden=flood_zones,
    geology=geology,
    water_crossings=water_crossings,
    city_boundary=city_boundary,
    base_config=config,
)

summary = wm.scenario_summary_table(scenarios, demand)
summary.assign(
    length_km=lambda df: df["length_km"].round(1),
    served_share=lambda df: (100 * df["served_share"]).round(1),
    avg_station_spacing_m=lambda df: df["avg_station_spacing_m"].round(0),
    min_station_spacing_m=lambda df: df["min_station_spacing_m"].round(0),
    max_station_spacing_m=lambda df: df["max_station_spacing_m"].round(0),
    avg_geology_factor=lambda df: df["avg_geology_factor"].round(2),
    geology_excess_km=lambda df: df["geology_excess_km"].round(2),
    high_geology_km=lambda df: df["high_geology_km"].round(2),
    anchor_geology_penalty=lambda df: df["anchor_geology_penalty"].round(0),
    station_geology_penalty=lambda df: df["station_geology_penalty"].round(0),
    station_flood_zone_count=lambda df: df["station_flood_zone_count"].astype(int),
    station_water_zone_count=lambda df: df["station_water_zone_count"].astype(int),
    station_flood_buffer_count=lambda df: df["station_flood_buffer_count"].astype(int),
    station_water_buffer_count=lambda df: df["station_water_buffer_count"].astype(int),
    forbidden_km=lambda df: df["forbidden_km"].round(2),
    water_crossing_km=lambda df: df["water_crossing_km"].round(2),
    outside_city_km=lambda df: df["outside_city_km"].round(2),
    outside_city_penalty=lambda df: df["outside_city_penalty"].round(0),
    line_overlap_km=lambda df: df["line_overlap_km"].round(2),
    max_turn_angle_deg=lambda df: df["max_turn_angle_deg"].round(0),
    mean_turn_angle_deg=lambda df: df["mean_turn_angle_deg"].round(0),
    curve_radius_violation_m=lambda df: df["curve_radius_violation_m"].round(0),
    corridor_detour_ratio=lambda df: df["corridor_detour_ratio"].round(2),
    corridor_backtrack_km=lambda df: df["corridor_backtrack_km"].round(2),
    corridor_shape_penalty=lambda df: df["corridor_shape_penalty"].round(0),
    turn_penalty=lambda df: df["turn_penalty"].round(0),
    transfer_score=lambda df: df["transfer_score"].round(0),
    base_cost_mln=lambda df: df["base_cost_mln"].round(0),
    flood_extra_cost_mln=lambda df: df["flood_extra_cost_mln"].round(0),
    estimated_cost_mln=lambda df: df["estimated_cost_mln"].round(0),
)

W scenariuszach wieloliniowych algorytm nie planuje każdej nitki w izolacji. Po zbudowaniu pierwszej linii popyt w jej zasięgu jest zmniejszany jako popyt resztkowy, więc druga linia szuka obszarów nadal słabo obsłużonych. Dodatkowo kolejne linie dostają premię za sensowne przecięcie z wcześniejszymi liniami albo dojście w pobliże ich stacji, ale jednocześnie dostają karę za długie prowadzenie równolegle w tym samym korytarzu.


Poniżej widać dziennik decyzji heurystyki: który kandydat został wstawiony, na którą pozycję, jaki był przyrost wyniku i ile metrów trasy to dodało. To jest miejsce, w którym w raporcie można pokazać, że algorytm realnie rozwiązuje problem wyboru i kolejności stacji, a nie tylko rysuje linię.


In [ ]:
scenarios[selected_scenario if "selected_scenario" in globals() else "1"]["candidates"].head(15)

In [ ]:
selected_scenario = "3"  # zmień na "1", "2" albo "3"

fig, ax = wm.plot_scenario(
    demand=demand,
    forbidden=flood_zones,
    scenario=scenarios[selected_scenario],
    centres=centres,
    regional_centres=regional_centres,
    regional_clusters=regional_clusters,
    geology=geology,
    demand_areas=demand_areas,
    water_crossings=water_crossings,
    city_boundary=city_boundary,
    centre_area=centre_area,
    figsize=(10, 9),
)
plt.show()

In [ ]:
fig, ax = wm.plot_scenario_satellite(
    demand=demand,
    forbidden=flood_zones,
    scenario=scenarios[selected_scenario],
    centres=centres,
    regional_centres=regional_centres,
    regional_clusters=regional_clusters,
    geology=geology,
    demand_areas=demand_areas,
    water_crossings=water_crossings,
    city_boundary=city_boundary,
    centre_area=centre_area,
    figsize=(11, 10),
)
plt.show()

## Mapy satelitarne scenariuszy

Na finalnych mapach scenariuszy gęstość zaludnienia zostaje półprzezroczysta, a linie metra są rysowane jako mocne, nieprzezroczyste linie z czarną obwódką. Dzięki temu linia nie ginie na zdjęciu satelitarnym ani na ciemniejszych obszarach zagęszczenia.


In [ ]:
for scenario_id, scenario in scenarios.items():
    fig, ax = wm.plot_scenario_satellite(
        demand=demand,
        forbidden=flood_zones,
        scenario=scenario,
        centres=centres,
        regional_centres=regional_centres,
        regional_clusters=regional_clusters,
        geology=geology,
        demand_areas=demand_areas,
        water_crossings=water_crossings,
        city_boundary=city_boundary,
        centre_area=centre_area,
        figsize=(9, 8),
    )
    ax.set_title(f"Scenariusz {scenario_id}: {scenario['name']}")
    plt.show()

## Kotwice wybrane przez algorytm

Kotwice to punkty kontrolne, ktore heurystyka wybrala i ulozyla w przebieg korytarza. Nie sa one tym samym co wszystkie stacje. Mniejsza liczba kotwic ogranicza efekt zygzaka, a kara za ostre skrety pilnuje, zeby linia nie wygrywala tylko dlatego, ze agresywnie skacze miedzy lokalnymi skupiskami popytu.


In [ ]:
anchors = pd.concat([line["anchors"] for line in scenarios[selected_scenario]["lines"]], ignore_index=True)
anchors_wgs = gpd.GeoDataFrame(anchors, geometry="geometry", crs=anchors.crs).to_crs(4326)
anchors_wgs["lon"] = anchors_wgs.geometry.x
anchors_wgs["lat"] = anchors_wgs.geometry.y
anchors_wgs[[
    "line_id",
    "anchor_order",
    "candidate_id",
    "name",
    "source",
    "candidate_weight",
    "anchor_relocated_m",
    "in_flood_zone",
    "in_flood_buffer",
    "lon",
    "lat",
]].head(40)

## Tabela stacji dla wybranego scenariusza

Stacje sa rozmieszczane adaptacyjnie na wybranym korytarzu. Nadal jest ich tyle samo co na M1, ale algorytm pilnuje minimalnego i maksymalnego odstepu, obniza atrakcyjnosc miejsc w obszarach zalewowych i dodaje bonus przy kotwicach korytarza. Dzieki temu zalamania trasy, jezeli zostaja, czesciej wypadaja przy sensownym miejscu stacyjnym zamiast posrodku pustego odcinka.


In [ ]:
lines, stations = wm.scenario_geodataframes(scenarios[selected_scenario])
stations_wgs = stations.to_crs(4326).copy()
stations_wgs["lon"] = stations_wgs.geometry.x
stations_wgs["lat"] = stations_wgs.geometry.y

stations_wgs[[
    "line_id",
    "station_code",
    "distance_m",
    "spacing_from_previous_m",
    "spacing_to_next_m",
    "station_placement",
    "near_route_anchor",
    "nearest_route_anchor_m",
    "lon",
    "lat",
]].head(30)

## Napotkane problemy i rozwiazania

1. Problem punktow zamiast obszarow: pierwsza wersja traktowala popyt jako punkty. Rozwiazanie: popyt jest teraz wczytywany i rysowany jako obszary demograficzne SIP, a punkty reprezentatywne sa uzywane tylko w algorytmie odleglosciowym.
2. Problem rzeki jako zakazu: woda powierzchniowa nie powinna blokowac metra tak jak brak gruntu dla zwyklej drogi. Rozwiazanie: rzeka jest osobna warstwa bariery komunikacyjnej; przeciecie tej bariery moze dostac premie, bo tworzy nowe polaczenie miedzy brzegami.
3. Problem zalewowosci: obszary ryzyka nie powinny przyciagac stacji, ale popyt z ich okolic nie powinien znikac. Rozwiazanie: wagi popytu z obszarow zakazanych sa przesuwane do najblizszego wolnego miejsca, a budowa przez MZP ma koszt x2.
4. Problem izolowanych i nakladajacych sie linii: kolejne nitki metra nie moga byc planowane tak, jakby wczesniejsze nie istnialy, ale nie powinny tez lezec na sobie ani jechac caly czas do tego samego ramienia miasta. Rozwiazanie: po kazdej linii mocniej zmniejszamy popyt resztkowy w jej zasiegu, premiujemy sensowne przesiadki i karzemy dlugi przebieg w szerszym buforze juz zaplanowanego korytarza.
5. Problem NP-trudnosci: pelne przeszukanie kombinacji i kolejnosci stacji szybko staje sie nierealne. Rozwiazanie: dla malej probki pokazujemy brute force, a dla pelnego problemu uzywamy heurystyki greedy insertion + 2-opt.
6. Problem interpretacji wyniku: kotwice algorytmu nie sa jeszcze finalna lista stacji. Rozwiazanie: rozdzielamy `anchor_objective_score` od `final_score`, czyli wynik wyboru korytarza od wyniku po rozlozeniu 21 stacji.
7. Problem zygzakow: wczesniejsza wersja pozwalala jednej linii odwiedzac zbyt wiele punktow jak trasie TSP, wiec geometria byla zbyt lamana. Rozwiazanie: ograniczamy liczbe kotwic korytarza, dodajemy mocniejsza kare za ostre skrety, twardy limit bardzo ostrych zalaman, kare za objazdowosc/backtracking korytarza i bonus stacyjny przy kotwicach, zeby zakrety nie byly przypadkowymi punktami na mapie.
8. Problem wychodzenia poza miasto: przy stalej dlugosci 23,1 km konce linii mogly byc wypychane poza granice Wroclawia. Rozwiazanie: budujemy `city_boundary` z granic osiedli SIP i karzemy kazdy kilometr trasy poza miastem przez `outside_city_penalty_per_km`.
9. Problem geologii jako nieczytelnego mnoznika: sam sredni `cost_factor` nie mowil, ile trudnego terenu faktycznie przecina linia. Rozwiazanie: dodajemy realna warstwe `data/raw/geology.geojson`, liczymy `geology_excess_km` po dlugosci trasy, osobno `high_geology_km` dla `cost_factor >= 1.5` oraz karzemy stacje i kotwice lezace bezposrednio na trudnym gruncie.
10. Problem stacji na wodzie lub wysokim ryzyku: tunel moze przejsc pod woda, ale stacja nie powinna wypasc na zbiorniku ani w buforze MZP. Rozwiazanie: wody powierzchniowe sa warstwa unikania dla kandydatow i stacji, a koncowe stacje moga odsunac sie od samego konca korytarza, jezeli ten koniec wypada w zlym miejscu.


## Jak rozwijac model

Najblizsze sensowne kroki:

1. Pobrac demografie SIP Wroclawia 1998-2025 i uzyc rejonow statystycznych jako podstawowego popytu.
2. Pobrac obszary zagrozenia powodziowego MZP z Hydroportalu i zapisac jako `data/raw/flood_zones.geojson` albo shapefile.
3. Uzupelnic `geology` o realny mnoznik kosztu: odwierty, grunty, poziom wod gruntowych, tunele pod rzeka.
4. Dodac aktualne wezly transportowe: dworce, petle tramwajowe, duze przesiadki autobusowe, uczelnie, szpitale, duze osiedla.
5. Zastapic lamane korytarze geometria krzywych albo grafem kandydatow: korytarze uliczne, kolejowe, podziemne i zakazane poligony jako koszt przejscia.
6. Dodac walidacje: ile mieszkancow ma stacje w 600, 800 i 1000 m oraz ile obecnych podrozy tramwaj/autobus moze zostac przejetych.
